# Partitioning and Compaction strategies

In [ ]:
import os
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from delta.tables import *

MINIO_ACCESS_KEY = "pkIeKAn4xpjoOgXiHQPw"
MINIO_SECRET_KEY = "JZRBfRszxLZzPeaAQaEXk32JxUKv25DVjUoO06Rk"
DATABASE = "default"
BUCKET_BRONZE = "bronze"
BUCKET_SILVER = "silver"
BUCKET_GOLD = "gold"

In [ ]:
%%time
spark = SparkSession.builder.master("spark://spark-master:7077") \
    .appName("MyAppClass03") \
    .config("spark.eventLog.enabled", "true") \
    .config("spark.eventLog.dir", "file:/tmp/spark-logs") \
    .config("spark.history.fs.logDirectory", "file:/tmp/spark-logs") \
    .config("log4j.rootCategory", "INFO, console") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,io.delta:delta-core_2.12:2.4.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.executor.instances", "4") \
    .config("spark.executor.cores", "2") \
    .config("spark.executor.memory", "1536m") \
    .config("spark.driver.memory", "1536m") \
    .config("spark.sql.shuffle.partitions", "16") \
    .config("spark.storage.memoryFraction", "0.4") \
    .config("spark.shuffle.memoryFraction", "0.5") \
    .config("spark.memory.offHeap.enabled", "true") \
    .config("spark.memory.offHeap.size", "512m") \
    .config("spark.sql.parquet.compression.codec", "gzip") \
    .config("spark.sql.orc.compression.codec", "zlib") \
    .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \
    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
    .config("spark.cleaner.referenceTracking.cleanCheckpoints", "true") \
    .config("spark.executor.cleanupOnShutdown", "true") \
    .getOrCreate()

In [ ]:
sc = spark.sparkContext
hadoop_conf = sc._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.access.key", MINIO_ACCESS_KEY)
hadoop_conf.set("fs.s3a.secret.key", MINIO_SECRET_KEY)
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

In [ ]:
spark

## PROJECT 1: HOTEL BOOKING

- [Data source](https://www.kaggle.com/datasets/mojtaba142/hotel-booking)

PARTITION

1. Reading staging
2. Table partitioning rules
3. Choice the best partition by column
4. Write data into partitioned bronze
5. Let's write using a bad partitioning strategy
6. Now, let's write using a good partitioning strategy
7. Looking at Trade-off solutions and describing the results

COMPACTION STRATEGIES

1. Writing data into the bronze, generating intentionally small files in a new table
2. Looking at Trade-off solutions and describing the results

## PARTITION

### 1 - READING STAGING (RAW)

In [ ]:
location_raw = f"s3a://staging"
file = "hotel_booking.csv"
data_origen = f"{location_raw}/{file}"

In [ ]:
df = spark.read.format('csv').option('header', 'true').option('inferSchema', 'true').load(data_origen)
df = df.withColumnRenamed("phone-number", "phone_number")

In [ ]:
df.createOrReplaceTempView("staging")

### 2 - Table partitioning rules

1. If your table is smaller than 1 TB, don’t add partitions; just use OPTIMIZE to reduce the
number of files.

### 3 - Choice the best partition by column
   
1. Is the cardinality of a column very high?

    If so, do not use that column for partitioning. For example, if you partition by a
    column `userId` and there can be more than a million distinct user IDs, then that
    is a bad partitioning strategy.

2. Is the balanced distribution?

    The data should be distributed evenly among partitions to prevent some from becoming too large while others remain nearly empty.

3. How often are filters?

    The column should be commonly used in WHERE clauses in queries to leverage the benefits of partitioning and enhance performance.

4. How much data will exist in each partition?

    You can partition by a column if you expect data in that partition to be at least
    1 GB.

In [ ]:
spark.sql(f"""
    SELECT
      COUNT(DISTINCT name) AS unique_name_values,
      COUNT(DISTINCT email) AS unique_email_values,
      COUNT(DISTINCT credit_card) AS unique_credit_card_values,
      COUNT(DISTINCT hotel) AS unique_hotel_values,
      COUNT(DISTINCT is_canceled) AS unique_is_canceled_values,
      COUNT(DISTINCT lead_time) AS unique_lead_time_values,
      COUNT(DISTINCT arrival_date_year) AS unique_arrival_date_year_values,
      COUNT(DISTINCT arrival_date_month) AS unique_arrival_date_month_values,
      COUNT(DISTINCT arrival_date_week_number) AS unique_arrival_date_week_number_values,
      COUNT(DISTINCT arrival_date_day_of_month) AS unique_arrival_date_day_of_month_values,
      COUNT(DISTINCT stays_in_weekend_nights) AS unique_stays_in_weekend_nights_values,
      COUNT(DISTINCT stays_in_week_nights) AS unique_stays_in_week_nights_values,
      COUNT(DISTINCT adults) AS unique_adults_values,
      COUNT(DISTINCT children) AS unique_children_values,
      COUNT(DISTINCT babies) AS unique_babies_values,
      COUNT(DISTINCT meal) AS unique_meal_values,
      COUNT(DISTINCT country) AS unique_country_values,
      COUNT(DISTINCT market_segment) AS unique_market_segment_values,
      COUNT(DISTINCT distribution_channel) AS unique_distribution_channel_values,
      COUNT(DISTINCT is_repeated_guest) AS unique_is_repeated_guest_values,
      COUNT(DISTINCT previous_cancellations) AS unique_previous_cancellations_values,
      COUNT(DISTINCT previous_bookings_not_canceled) AS unique_previous_bookings_not_canceled_values,
      COUNT(DISTINCT reserved_room_type) AS unique_reserved_room_type_values,
      COUNT(DISTINCT assigned_room_type) AS unique_assigned_room_type_values,
      COUNT(DISTINCT booking_changes) AS unique_booking_changes_values,
      COUNT(DISTINCT deposit_type) AS unique_deposit_type_values,
      COUNT(DISTINCT agent) AS unique_agent_values,
      COUNT(DISTINCT company) AS unique_company_values,
      COUNT(DISTINCT days_in_waiting_list) AS unique_days_in_waiting_list_values,
      COUNT(DISTINCT customer_type) AS unique_customer_type_values,
      COUNT(DISTINCT adr) AS unique_adr_values,
      COUNT(DISTINCT required_car_parking_spaces) AS unique_required_car_parking_spaces_values,
      COUNT(DISTINCT total_of_special_requests) AS unique_total_of_special_requests_values,
      COUNT(DISTINCT reservation_status) AS unique_reservation_status_values,
      COUNT(DISTINCT reservation_status_date) AS unique_reservation_status_date_values
    FROM staging;
""").show(truncate=False)

In [ ]:
# +------------------+-------------------+-------------------------+-------------------+-------------------------+-----------------------+-------------------------------+--------------------------------+--------------------------------------+---------------------------------------+-------------------------------------+----------------------------------+--------------------+----------------------+--------------------+------------------+---------------------+----------------------------+----------------------------------+-------------------------------+------------------------------------+--------------------------------------------+--------------------------------+--------------------------------+-----------------------------+--------------------------+-------------------+---------------------+----------------------------------+---------------------------+-----------------+-----------------------------------------+---------------------------------------+--------------------------------+-------------------------------------+
# |unique_name_values|unique_email_values|unique_credit_card_values|unique_hotel_values|unique_is_canceled_values|unique_lead_time_values|unique_arrival_date_year_values|unique_arrival_date_month_values|unique_arrival_date_week_number_values|unique_arrival_date_day_of_month_values|unique_stays_in_weekend_nights_values|unique_stays_in_week_nights_values|unique_adults_values|unique_children_values|unique_babies_values|unique_meal_values|unique_country_values|unique_market_segment_values|unique_distribution_channel_values|unique_is_repeated_guest_values|unique_previous_cancellations_values|unique_previous_bookings_not_canceled_values|unique_reserved_room_type_values|unique_assigned_room_type_values|unique_booking_changes_values|unique_deposit_type_values|unique_agent_values|unique_company_values|unique_days_in_waiting_list_values|unique_customer_type_values|unique_adr_values|unique_required_car_parking_spaces_values|unique_total_of_special_requests_values|unique_reservation_status_values|unique_reservation_status_date_values|
# +------------------+-------------------+-------------------------+-------------------+-------------------------+-----------------------+-------------------------------+--------------------------------+--------------------------------------+---------------------------------------+-------------------------------------+----------------------------------+--------------------+----------------------+--------------------+------------------+---------------------+----------------------------+----------------------------------+-------------------------------+------------------------------------+--------------------------------------------+--------------------------------+--------------------------------+-----------------------------+--------------------------+-------------------+---------------------+----------------------------------+---------------------------+-----------------+-----------------------------------------+---------------------------------------+--------------------------------+-------------------------------------+
# |81503             |115889             |9000                     |2                  |2                        |479                    |3                              |12                              |53                                    |31                                     |17                                   |35                                |14                  |5                     |5                   |5                 |177                  |8                           |5                                 |2                              |15                                  |73                                          |10                              |12                              |21                           |3                         |333                |352                  |128                               |4                          |8879             |5                                        |6                                      |3                               |926                                  |
# +------------------+-------------------+-------------------------+-------------------+-------------------------+-----------------------+-------------------------------+--------------------------------+--------------------------------------+---------------------------------------+-------------------------------------+----------------------------------+--------------------+----------------------+--------------------+------------------+---------------------+----------------------------+----------------------------------+-------------------------------+------------------------------------+--------------------------------------------+--------------------------------+--------------------------------+-----------------------------+--------------------------+-------------------+---------------------+----------------------------------+---------------------------+-----------------+-----------------------------------------+---------------------------------------+--------------------------------+-------------------------------------+

In [ ]:
spark.sql("SELECT * FROM staging LIMIT 10").show(truncate=False)

In [ ]:
# +------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+-----+---------------------------+-------------------------+------------------+-----------------------+----------------+---------------------------+------------+----------------+
# |hotel       |is_canceled|lead_time|arrival_date_year|arrival_date_month|arrival_date_week_number|arrival_date_day_of_month|stays_in_weekend_nights|stays_in_week_nights|adults|children|babies|meal|country|market_segment|distribution_channel|is_repeated_guest|previous_cancellations|previous_bookings_not_canceled|reserved_room_type|assigned_room_type|booking_changes|deposit_type|agent|company|days_in_waiting_list|customer_type|adr  |required_car_parking_spaces|total_of_special_requests|reservation_status|reservation_status_date|name            |email                      |phone_number|credit_card     |
# +------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+-----+---------------------------+-------------------------+------------------+-----------------------+----------------+---------------------------+------------+----------------+
# |Resort Hotel|0          |342      |2015             |July              |27                      |1                        |0                      |0                   |2     |0.0     |0     |BB  |PRT    |Direct        |Direct              |0                |0                     |0                             |C                 |C                 |3              |No Deposit  |null |null   |0                   |Transient    |0.0  |0                          |0                        |Check-Out         |2015-07-01             |Ernest Barnes   |Ernest.Barnes31@outlook.com|669-792-1661|************4322|
# |Resort Hotel|0          |737      |2015             |July              |27                      |1                        |0                      |0                   |2     |0.0     |0     |BB  |PRT    |Direct        |Direct              |0                |0                     |0                             |C                 |C                 |4              |No Deposit  |null |null   |0                   |Transient    |0.0  |0                          |0                        |Check-Out         |2015-07-01             |Andrea Baker    |Andrea_Baker94@aol.com     |858-637-6955|************9157|
# |Resort Hotel|0          |7        |2015             |July              |27                      |1                        |0                      |1                   |1     |0.0     |0     |BB  |GBR    |Direct        |Direct              |0                |0                     |0                             |A                 |C                 |0              |No Deposit  |null |null   |0                   |Transient    |75.0 |0                          |0                        |Check-Out         |2015-07-02             |Rebecca Parker  |Rebecca_Parker@comcast.net |652-885-2745|************3734|
# |Resort Hotel|0          |13       |2015             |July              |27                      |1                        |0                      |1                   |1     |0.0     |0     |BB  |GBR    |Corporate     |Corporate           |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |304.0|null   |0                   |Transient    |75.0 |0                          |0                        |Check-Out         |2015-07-02             |Laura Murray    |Laura_M@gmail.com          |364-656-8427|************5677|
# |Resort Hotel|0          |14       |2015             |July              |27                      |1                        |0                      |2                   |2     |0.0     |0     |BB  |GBR    |Online TA     |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |240.0|null   |0                   |Transient    |98.0 |0                          |1                        |Check-Out         |2015-07-03             |Linda Hines     |LHines@verizon.com         |713-226-5883|************5498|
# |Resort Hotel|0          |14       |2015             |July              |27                      |1                        |0                      |2                   |2     |0.0     |0     |BB  |GBR    |Online TA     |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |240.0|null   |0                   |Transient    |98.0 |0                          |1                        |Check-Out         |2015-07-03             |Jasmine Fletcher|JFletcher43@xfinity.com    |190-271-6743|************9263|
# |Resort Hotel|0          |0        |2015             |July              |27                      |1                        |0                      |2                   |2     |0.0     |0     |BB  |PRT    |Direct        |Direct              |0                |0                     |0                             |C                 |C                 |0              |No Deposit  |null |null   |0                   |Transient    |107.0|0                          |0                        |Check-Out         |2015-07-03             |Dylan Rangel    |Rangel.Dylan@comcast.net   |420-332-5209|************6994|
# |Resort Hotel|0          |9        |2015             |July              |27                      |1                        |0                      |2                   |2     |0.0     |0     |FB  |PRT    |Direct        |Direct              |0                |0                     |0                             |C                 |C                 |0              |No Deposit  |303.0|null   |0                   |Transient    |103.0|0                          |1                        |Check-Out         |2015-07-03             |William Velez   |Velez_William@mail.com     |286-669-4333|************8729|
# |Resort Hotel|1          |85       |2015             |July              |27                      |1                        |0                      |3                   |2     |0.0     |0     |BB  |PRT    |Online TA     |TA/TO               |0                |0                     |0                             |A                 |A                 |0              |No Deposit  |240.0|null   |0                   |Transient    |82.0 |0                          |1                        |Canceled          |2015-05-06             |Steven Murphy   |Steven.Murphy54@aol.com    |341-726-5787|************3639|
# |Resort Hotel|1          |75       |2015             |July              |27                      |1                        |0                      |3                   |2     |0.0     |0     |HB  |PRT    |Offline TA/TO |TA/TO               |0                |0                     |0                             |D                 |D                 |0              |No Deposit  |15.0 |null   |0                   |Transient    |105.5|0                          |0                        |Canceled          |2015-04-22             |Michael Moore   |MichaelMoore81@outlook.com |316-648-6176|************9190|
# +------------+-----------+---------+-----------------+------------------+------------------------+-------------------------+-----------------------+--------------------+------+--------+------+----+-------+--------------+--------------------+-----------------+----------------------+------------------------------+------------------+------------------+---------------+------------+-----+-------+--------------------+-------------+-----+---------------------------+-------------------------+------------------+-----------------------+----------------+---------------------------+------------+----------------+

### 4 - Write data into partitioned bronze

In [ ]:
table_bronze_bad = "hotel_booking_partitioned_bronze_bad"
table_bronze_good = "hotel_booking_partitioned_bronze_good"
location_bronze_bad = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze_bad}"
location_bronze_good = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze_good}"

### 5 - Let's write using a bad partitioning strategy

In [ ]:
%%time
# Writing in Delta format
(
    df.write.format("delta")
    .mode("overwrite")
    .partitionBy("reservation_status_date")
    .save(location_bronze_bad)
)

### 6 - Now, let's write using a good partitioning strategy

In [ ]:
%%time
(
    df.write.format("delta")
    .mode("overwrite")
    .partitionBy("arrival_date_year", "arrival_date_month")
    .save(location_bronze_good)
)

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_bronze_bad}
    USING DELTA
    LOCATION '{location_bronze_bad}'
""")

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_bronze_good}
    USING DELTA
    LOCATION '{location_bronze_good}'
""")

### 7 - Looking at Trade-off solutions and describing the results

In [ ]:
%%time
spark.sql(f"SELECT COUNT(*) FROM {DATABASE}.{table_bronze_bad} WHERE reservation_status_date = '2015-12-15'").show()

In [ ]:
%%time
spark.sql(f"""
SELECT COUNT(*) 
FROM {DATABASE}.{table_bronze_good} 
WHERE 
    arrival_date_year = '2015'
    AND arrival_date_month = 'August'
""").show(truncate=False)

In [ ]:
spark.sql(f"""
DESCRIBE DETAIL {DATABASE}.{table_bronze_bad} 
""").show(truncate=False)

In [ ]:
# +------+------------------------------------+----------------------------------------------------------+-----------+-------------------------------------------------------+-----------------------+-------------------+-------------------------+--------+-----------+----------+----------------+----------------+------------------------+
# |format|id                                  |name                                                      |description|location                                               |createdAt              |lastModified       |partitionColumns         |numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|tableFeatures           |
# +------+------------------------------------+----------------------------------------------------------+-----------+-------------------------------------------------------+-----------------------+-------------------+-------------------------+--------+-----------+----------+----------------+----------------+------------------------+
# |delta |562e8229-ea62-4246-9637-666de0f223be|spark_catalog.default.hotel_booking_partitioned_bronze_bad|null       |s3a://bronze/delta/hotel_booking_partitioned_bronze_bad|2024-11-18 16:46:52.683|2024-11-18 16:47:20|[reservation_status_date]|2818    |38793296   |{}        |1               |2               |[appendOnly, invariants]|
# +------+------------------------------------+----------------------------------------------------------+-----------+-------------------------------------------------------+-----------------------+-------------------+-------------------------+--------+-----------+----------+----------------+----------------+------------------------+

In [ ]:
spark.sql(f"""
DESCRIBE DETAIL {DATABASE}.{table_bronze_good} 
""").show(truncate=False)

In [ ]:
# +------+------------------------------------+-----------------------------------------------------------+-----------+--------------------------------------------------------+-----------------------+-------------------+---------------------------------------+--------+-----------+----------+----------------+----------------+------------------------+
# |format|id                                  |name                                                       |description|location                                                |createdAt              |lastModified       |partitionColumns                       |numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|tableFeatures           |
# +------+------------------------------------+-----------------------------------------------------------+-----------+--------------------------------------------------------+-----------------------+-------------------+---------------------------------------+--------+-----------+----------+----------------+----------------+------------------------+
# |delta |b3834289-8d57-4105-83e5-7b0aeef14809|spark_catalog.default.hotel_booking_partitioned_bronze_good|null       |s3a://bronze/delta/hotel_booking_partitioned_bronze_good|2024-11-18 16:47:29.666|2024-11-18 16:47:33|[arrival_date_year, arrival_date_month]|86      |5092218    |{}        |1               |2               |[appendOnly, invariants]|
# +------+------------------------------------+-----------------------------------------------------------+-----------+--------------------------------------------------------+-----------------------+-------------------+---------------------------------------+--------+-----------+----------+----------------+----------------+------------------------+

## COMPACTION STRATEGIES

### 1 - Rewriting data into the bronze, generating intentionally small files

We can easily create way too many small files using the normal `hotel_booking` table as our source. The total number of rows in the table is **119.390**. If we repartition the table to, say, two thousand partitions, this will split the table into an even two thousand files at around **15 kb** per file

In [ ]:
table_bronze = "hotel_booking_compaction"
location_bronze = f"s3a://{BUCKET_BRONZE}/delta/{table_bronze}"

In [ ]:
%%time
(
    df.repartition(2000)
    .write
    .format("delta")
    .mode("overwrite")
    .save(location_bronze)
)

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {DATABASE}.{table_bronze}
    USING DELTA
    LOCATION '{location_bronze}'
""")

### 2 - Looking at Trade-off solutions and describing the results

In [ ]:
spark.sql(f"""
DESCRIBE DETAIL {DATABASE}.{table_bronze} 
""").show(truncate=False)

In [ ]:
# +------+------------------------------------+----------------------------------------------+-----------+-------------------------------------------+-----------------------+-------------------+----------------+--------+-----------+----------+----------------+----------------+------------------------+
# |format|id                                  |name                                          |description|location                                   |createdAt              |lastModified       |partitionColumns|numFiles|sizeInBytes|properties|minReaderVersion|minWriterVersion|tableFeatures           |
# +------+------------------------------------+----------------------------------------------+-----------+-------------------------------------------+-----------------------+-------------------+----------------+--------+-----------+----------+----------------+----------------+------------------------+
# |delta |236549aa-a423-4a97-945b-9c0dfdee5c61|spark_catalog.default.hotel_booking_compaction|null       |s3a://bronze/delta/hotel_booking_compaction|2024-11-16 02:57:06.765|2024-11-16 02:59:16|[]              |2000    |31136934   |{}        |1               |2               |[appendOnly, invariants]|
# +------+------------------------------------+----------------------------------------------+-----------+-------------------------------------------+-----------------------+-------------------+----------------+--------+-----------+----------+----------------+----------------+------------------------+

For tuning the OPTIMIZE thresholds, there are a few considerations to keep in mind:

- (Spark only) `spark.databricks.delta.optimize.minFileSize` (long) is used to group together files smaller than the threshold (in bytes) before being rewritten into a larger file by the OPTIMIZE command.

- (Spark only) `spark.databricks.delta.optimize.maxFileSize` (long) is used to
specify the target file size produced by the OPTIMIZE command.

- (Spark only) `spark.databricks.delta.optimize.repartition.enabled` (bool): is used to change the behavior of OPTIMIZE and will use repartition(1) instead of coalesce(1) when reducing.

- (delta-rs) The table property `delta.targetFileSize` (string)—an example being 250mb—can be used with the delta-rs client but is currently not supported in the OSS delta release.

_Reference_

_Delta Lake: The Definitive Guide_

<div style="background-color: #f9f9f9; border-left: 6px solid #ffa94f; padding: 10px; margin: 10px 0;">

**Note**: Remind it that there are risks while we make this type of compaction, therefore, ensure that you have a storage redundancy for the table:

- Computational Cost
- Configuration Complexity
- **Read and Write Downtime Window**
- Possibility of Data Loss

</div>

In [ ]:
%%time
(
    DeltaTable
    .forName(spark, f"{DATABASE}.{table_bronze}")
    .optimize()
    .executeCompaction()
)

In [ ]:
(
    DeltaTable.forName(spark, f"{DATABASE}.{table_bronze}")
    .history(10)
    .where(F.col("operation") == "OPTIMIZE")
    .select(
    "version", "timestamp", "operation",
    "operationMetrics.numRemovedFiles",
    "operationMetrics.numAddedFiles"
    )
    .show(truncate=False)
)

In [ ]:
spark.stop()